# AI-Powered Promotional Copy Generation - Case Study

Welcome to the case study! Your goal is to build an automated pipeline that takes promotion data and generates tailored marketing copy using Large Language Models (LLMs).

### Requirements:
1. **Input Data**: Read the `data/promotions.csv`.
2. **Style Guides**: Read the corresponding style guidelines from the `styles/` folder for each client.
3. **Generation**: For **each item** in the CSV, generate **3 examples**. Each example must include:
   - A `simplified` version of the copy.
   - A `complicated` version of the copy.
4. **LLM API**: Use the provided `llm.chat()` function from `src.llm` to simulate LLM calls.
5. **Verification (Optional/Bonus)**: Write a verification step to ensure the generated output follows the constraints in the `.md` files (e.g., price is mentioned, length limit is respected).

In [ ]:
import pandas as pd
import json
import os
import sys

# Allow importing from src
sys.path.append('src')
from llm import llm

### 1. Load the Data & Style Guides

In [ ]:
# Load promotion data
df = pd.read_csv('data/promotions.csv')
display(df.head())

def load_style_guides(styles_dir):
    styles = {}
    for filename in os.listdir(styles_dir):
        if filename.endswith('_style.md'):
            client = filename.replace('_style.md', '')
            with open(os.path.join(styles_dir, filename), 'r') as f:
                styles[client] = f.read()
    return styles

styles = load_style_guides('styles')

### 2. Generate Promotional Copy (Testing One Item)
Let's test the workflow on the very first row.

In [ ]:
# Get the first row
row = df.iloc[0]
sku = row['sku']
client = row['client']
style_guide = styles.get(client, "")

print(f"Processing SKU: {sku} for Client: {client}\n")

# Build the Prompt for a Simplified Copy
prompt = f"""
You are an expert copywriter. Please write a SIMPLIFIED promotional copy for the following product.

Product Details:
- SKU: {sku}
- Title: {row['title']}
- Description: {row['description']}
- Original Price: {row['orig_price']}
- Promo Price: {row['promo_price']}
- Max Characters Allowed: {row['max_copy_chars']}

Brand Style Guide:
{style_guide}

Output ONLY the copy, nothing else.
"""

# Call the LLM (This will use Mock if Anthropic is not set up)
# generated_copy = llm.call(prompt)
generated_copy = llm.call([{"role": "user", "content": prompt}])
print("=== Generated Copy ===")
print(generated_copy)
print("======================")

### 3. Verification & Evaluation
Two-layer validation:
1. **Rule-based checks** — deterministic hard constraints (char limit, price, emoji, formatting).
2. **LLM-as-Judge** — subjective quality evaluation via `structured_chat` (tool-use), guaranteed JSON schema.

##### rule checks

In [ ]:
import re

# ── Layer 1: Rule-based checks ───────────────────────────────────
def rule_checks(copy_text, row, style_guide):
    results = {
        'length_ok': len(copy_text) <= row['max_copy_chars'],
        'char_count': len(copy_text),
        'max_allowed': int(row['max_copy_chars']),
        'price_mentioned': str(row['promo_price']) in copy_text,
    }
    # Client-specific rules from the style guides
    if row['client'] == 'TechCorp':
        results['price_bolded'] = (
            f"**{row['promo_price']}" in copy_text or
            f"**${row['promo_price']}" in copy_text
        )
        results['no_emoji'] = not bool(re.search(r'[\U0001F300-\U0001FAFF]', copy_text))
    elif row['client'] == 'FashionCo':
        results['has_emoji'] = bool(re.search(r'[\U0001F300-\U0001FAFF]', copy_text))
        results['price_has_exclamation'] = f"{row['promo_price']}!" in copy_text
    return results


# ── Layer 2: LLM-as-Judge (structured output via tool-use) ─────
JUDGE_SCHEMA = {
    "type": "object",
    "properties": {
        "tone_match": {
            "type": "object",
            "properties": {
                "score": {"type": "integer", "description": "1-5 scale, 5 = perfect tone match"},
                "reason": {"type": "string"}
            },
            "required": ["score", "reason"]
        },
        "language_correct": {
            "type": "object",
            "properties": {
                "score": {"type": "integer", "description": "1-5 scale, 5 = correct language (e.g. fr-CA for FashionCo)"},
                "reason": {"type": "string"}
            },
            "required": ["score", "reason"]
        },
        "formatting_ok": {
            "type": "object",
            "properties": {
                "score": {"type": "integer", "description": "1-5 scale, 5 = formatting fully compliant"},
                "reason": {"type": "string"}
            },
            "required": ["score", "reason"]
        },
        "completeness": {
            "type": "object",
            "properties": {
                "score": {"type": "integer", "description": "1-5 scale, 5 = all required info present"},
                "reason": {"type": "string"}
            },
            "required": ["score", "reason"]
        },
        "overall_pass": {"type": "boolean", "description": "true if the copy passes all criteria"},
        "suggestions": {"type": "string", "description": "Actionable improvements if any"}
    },
    "required": ["tone_match", "language_correct", "formatting_ok", "completeness", "overall_pass", "suggestions"]
}

JUDGE_PROMPT = '''\
You are a strict QA reviewer. Evaluate the promotional copy below against the brand style guide.

## Copy to evaluate
{copy_text}

## Brand style guide
{style_guide}

## Product info
- Promo price: {promo_price}
- Original price: {orig_price}
- Category: {category}

Score each criterion on a 1-5 scale (5 = perfect). Use the extract_data tool to return your evaluation.
'''


##### LLM-as-a-judge

In [ ]:
def llm_judge(copy_text, row, style_guide):
    prompt = JUDGE_PROMPT.format(
        copy_text=copy_text,
        style_guide=style_guide,
        promo_price=row['promo_price'],
        orig_price=row['orig_price'],
        category=row['category'],
    )
    return llm.structured_chat(prompt, JUDGE_SCHEMA, tool_name="extract_data")


# ── Combined verify_copy ───────────────────────────────────────
def verify_copy(copy_text, row, style_guide):
    result = {
        'rule_checks': rule_checks(copy_text, row, style_guide),
        'llm_judge': llm_judge(copy_text, row, style_guide),
    }
    return result

In [ ]:
# Run the two-layer verification on the copy we just generated
verification = verify_copy(generated_copy, row, style_guide)

print('\u2550' * 50)
print('  RULE-BASED CHECKS')
print('\u2550' * 50)
for k, v in verification['rule_checks'].items():
    status = '\u2705' if v is True else ('\u274c' if v is False else str(v))
    print(f'  {k:.<30s} {status}')

print()
print('\u2550' * 50)
print('  LLM-AS-JUDGE EVALUATION')
print('\u2550' * 50)
judge = verification['llm_judge']
for key in ['tone_match', 'language_correct', 'formatting_ok', 'completeness']:
    entry = judge[key]
    print(f"  {key:.<30s} {entry['score']}/5  \u2014 {entry['reason']}")
print(f"\n  Overall pass: {'\u2705' if judge['overall_pass'] else '\u274c'}")
print(f"  Suggestions:  {judge.get('suggestions', 'N/A')}")